In [3]:
import os
import torch
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from transformers import BitsAndBytesConfig
from langdetect import detect

In [4]:
# load embedding model
hf_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'},
    encode_kwargs={}
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
# load FAISS index
print("Loading FAISS index...")
vectorstore = FAISS.load_local(
    os.path.join("/content/drive/MyDrive/nepali-news-rag-chatbot/", "faiss_index_nepali"),
    hf_embeddings,
    allow_dangerous_deserialization=True
)

# set nprobe for IVF index (search 16 of 256 clusters)
vectorstore.index.nprobe = 16
print(f"Index loaded. {vectorstore.index.ntotal} vectors, nprobe={vectorstore.index.nprobe}")

Loading FAISS index...
Index loaded. 593779 vectors, nprobe=16


In [6]:
!hf auth login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: read).
The token `nepali-news-rag-app` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `nepal

In [7]:
# define model
model_id = "microsoft/Phi-3.5-mini-instruct"

In [8]:
# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [9]:
print(f"Loading {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
print("Model loaded.")

Loading microsoft/Phi-3.5-mini-instruct...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

Model loaded.


In [10]:
# create generation pipeline
text_generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,        # concise answers
    temperature=0.1,           # factual
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1,
    pad_token_id=tokenizer.eos_token_id  # avoid warnings
)

llm = HuggingFacePipeline(pipeline=text_generation_pipeline)
print("Pipeline ready.")

Device set to use cuda:0


Pipeline ready.


In [11]:
prompt_template = """<|system|>
You are a helpful AI assistant for Nepali News.
Use the following pieces of retrieved context to answer the user's question.
If the answer is not in the context, strictly say "I cannot find the answer in the provided news."
Keep the answer concise and factual.

Context:
{context}

Answer in the following language: {target_language}<|end|>
<|user|>
{question}<|end|>
<|assistant|>
"""

In [12]:
# setup retriever with k=15 (100-char chunks are small)
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 15}  # increased from 5 due to small chunk size
)

In [13]:
def get_answer(query, language=None):
    """
    Retrieve relevant docs and generate answer.

    Args:
        query: User question (English or Nepali)
        language: Target language ('English' or 'Nepali'). If None, auto-detect.

    Returns:
        (answer: str, sources: list)
    """
    # Input validation
    if not query or not query.strip():
        return "Please provide a question.", []

    # Auto-detect language if not specified
    if language is None:
        try:
            detected = detect(query)
            language = "English" if detected == "en" else "Nepali"
        except:
            language = "Nepali"  # default fallback

    # Retrieve relevant documents
    docs = retriever.invoke(query)

    # Check if retrieval found anything
    if not docs or len(docs) == 0:
        return "I don't have information on this topic.", []

    # Build context from retrieved chunks
    context_text = "\n\n".join([d.page_content for d in docs])

    # Format prompt
    final_prompt = prompt_template.format(
        context=context_text,
        question=query,
        target_language=language
    )

    # Generate response
    response = llm.invoke(final_prompt)

    # Extract assistant's answer (pipeline returns full prompt + generation)
    if "<|assistant|>" in response:
        response = response.split("<|assistant|>")[-1]

    # Strip special tokens
    response = response.replace("<|end|>", "").strip()

    # "I don't know" guardrail
    if "cannot find" in response.lower() or "don't have" in response.lower():
        response = "I cannot find the answer in the provided news."

    # Format sources (PRD P1 requirement)
    sources = [
        f"{d.metadata.get('source', 'Unknown')} - {d.metadata.get('heading', 'N/A')}"
        for d in docs
    ]

    return response, sources

In [14]:
# Example usage
query = "What are the recent developments in education?"
answer, sources = get_answer(query)

print("Answer:")
print(answer)
print("\nSources:")
for i, src in enumerate(sources[:3], 1):  # show top 3
    print(f"{i}. {src}")

Answer:
Based on the provided Nepali context, there have been several initiatives and plans regarding educational development recently:

1. **National Curriculum Development**: There has been an emphasis on developing new curricula with input from various stakeholders like teachers, parents, students, etc., aiming at improving national standards across different levels including primary school grades such as Grade 6-8. The focus seems to be on creating more engaging content that caters better to student needs while incorporating modern teaching methods.

2. **Vocational Training Expansion**: Efforts appear to be underway towards expanding vocational training programs within schools which could include practical skills alongside academic learning, potentially increasing employability post-graduation. This may involve partnerships between governmental bodies or private institutions offering specialized courses aligned with industry demands.

3. **Digital Learning Platforms**: With refere

In [16]:
# Example usage
query = "नेपाल को प्रधानमन्त्री को हुन्?"
answer, sources = get_answer(query)

print("Answer:")
print(answer)
print("\nSources:")
for i, src in enumerate(sources[:3], 1):  # show top 3
    print(f"{i}. {src}")

Answer:
नेपालमा प्रधानमन्त्री लोटेलाई भुटानका हुन्

Sources:
1. Educationpati - शिक्षामन्त्रीलाई संघको ध्यानाकर्षण पत्र : वजेटवृद्धि, स्थायीत्वदेखि तहगत बढुवाको माग
2. Ekantipur - गर्भावस्थामा मधुमेह
3. Nepalkhabar - भाजपा नेता स्वामी भन्छन् : केपी ओली भारतको रोजाई


In [17]:
# Example usage
query = "नेपालमा हालै भएका राजनीतिक परिवर्तनहरूको बारेमा यस लेखमा के भनिएको छ?"
answer, sources = get_answer(query)

print("Answer:")
print(answer)
print("\nSources:")
for i, src in enumerate(sources[:3], 1):  # show top 3
    print(f"{i}. {src}")

Answer:
यसैले राजनीतिक परिवर्तनहरूको बारेमा यस्तै आफ्नो ने प्रस्तुत किया राखेर डेढ घन्टासम्म पाप र पुण्यबारे प्रवचन दिने गरेका छन् ।

Sources:
1. Onlinekhabar - स्वस्तिमा लेख्छिन्–‘मेरो पेशामा कुनै प्रकारको उत्पीडन सह्य छैन’
2. Nepalkhabar - निर्वाचन आचारसंहिताप्रति एमाले उपमहासचिवको असन्तुष्टि
3. Onlinekhabar - हरिवंशले ल्याए पहिलो उपन्यास ‘हरिबहादुर’


In [18]:
# Example usage
query = "What does the article say about recent political changes in Nepal?"
answer, sources = get_answer(query)

print("Answer:")
print(answer)
print("\nSources:")
for i, src in enumerate(sources[:3], 1):  # show top 3
    print(f"{i}. {src}")

Answer:
The articles discuss various aspects of significant political shifts occurring in Nepal recently. Key points include opposition from seven farmers organizations against parliament dissolution by President Gajraj and Prime Minister Pushpa Kamal Dahal, with calls for diverse protests across different sectors as reactions to these actions. There have been concerns over alleged unconstitutionality, arbitrariness, insensitivity, and poor governance within politics. In terms of international relations, there seems to be an increasing influence of foreign powers like China through initiatives such as Belt and Road, which some view positively while others see it negatively impacting national sovereignty. Within internal dynamics, Narendra Modi emerges as a preferred ally according to Indian politician Subramanian Swamy. Meanwhile, local issues around agriculture sector growth indicate potential areas where private-public partnerships could foster development amidst challenges posed by 

In [19]:
# Example usage
query = "Who is the prime minister of Nepal?"
answer, sources = get_answer(query)

print("Answer:")
print(answer)
print("\nSources:")
for i, src in enumerate(sources[:3], 1):  # show top 3
    print(f"{i}. {src}")

Answer:
As per the given context, the current Prime Minister of Nepal is Khadga Prasad Sharma Oli. However, please note that political positions can change, and it's always good to verify from reliable sources for the most updated information.

Sources:
1. Nepalkhabar - भारतीय सेनाध्यक्ष पाँडे भदौ १९ गते नेपाल आउँदै, यस्तो छ भ्रमण तालिका
2. Educationpati - शिक्षामन्त्रीलाई संघको ध्यानाकर्षण पत्र : वजेटवृद्धि, स्थायीत्वदेखि तहगत बढुवाको माग
3. Nepalkhabar - राष्ट्रिय वाणिज्य बैंकका अध्यक्ष धनिराम शर्माद्वारा शपथ
